AuthorityEffect: None.


## Section 2: Daily Workflow Integrations

Practitioners can embed the Symbol Game directly into their existing development environments across three primary surfaces:
1. **Claude Code Hooks (`.claude/hooks/`)**: Intercept tool execution before side-effects occur.
2. **Codex / Copilot Linters**: Fail-closed boundary monitors inspecting generated code docstrings and type annotations.
3. **Antigravity Multi-Agent Swarms**: Sealed envelopes that prevent error cascades between collaborating agents.

Let us build and test each integration.


In [ ]:
# [Pinned] Core environment verification & dependencies
import hashlib
import json
import sys
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

# Verify runtime
assert sys.version_info >= (3, 10), "Requires Python >= 3.10"
print(f"[OK] Python version: {sys.version.split()[0]}")

In [ ]:
# [Pinned] Claude Code Hook Integration: Pre-Tool Execution Interceptor
# Blocks unauthorized or destructive tool actions before execution.


@dataclass(frozen=True)
class ToolInvocation:
    tool_name: str
    arguments: Dict[str, Any]


@dataclass(frozen=True)
class HookDecision:
    allowed: bool
    reason: str
    injected_guidance: Optional[str] = None


class ClaudeCodePreToolHook:
    """
    Simulates a Claude Code pre-tool interceptor (.claude/hooks/pre_tool_call).
    Enforces fail-closed safety and prevents self-certification or destructive shell actions.
    """

    FORBIDDEN_COMMAND_PATTERNS = ["rm -rf", "git reset --hard", "DROP TABLE", "git push --force"]
    PROTECTED_FILES = ["receipt.json", ".provenance", "AUTHORITY.asc"]

    @classmethod
    def evaluate(cls, call: ToolInvocation) -> HookDecision:
        if call.tool_name in ("run_command", "bash", "execute_command"):
            cmd = call.arguments.get("command", "")
            for pattern in cls.FORBIDDEN_COMMAND_PATTERNS:
                if pattern in cmd:
                    return HookDecision(
                        allowed=False,
                        reason=f"SECURITY_REFUSAL: Destructive pattern '{pattern}' detected in command.",
                        injected_guidance="Use safe inspect-only tools or request explicit user elevation.",
                    )
        elif call.tool_name in ("write_to_file", "edit_file"):
            target_path = call.arguments.get("path", "")
            for protected in cls.PROTECTED_FILES:
                if protected in target_path:
                    return HookDecision(
                        allowed=False,
                        reason=f"PROTECTED_FILE_REFUSAL: Cannot write directly to host-controlled file '{protected}'.",
                        injected_guidance="Host receipts and provenance files can only be written by Layer 3 host oracles.",
                    )
        return HookDecision(allowed=True, reason="Tool execution allowed under declared policies.")


# Test 1: Benign command
benign_call = ToolInvocation("run_command", {"command": "git status"})
decision_benign = ClaudeCodePreToolHook.evaluate(benign_call)
assert decision_benign.allowed is True
print(f"[OK] Benign call: {decision_benign.reason}")

# Test 2: Illicit destructive command blocked
malicious_call = ToolInvocation("run_command", {"command": "git reset --hard origin/main"})
decision_malicious = ClaudeCodePreToolHook.evaluate(malicious_call)
assert decision_malicious.allowed is False
assert "SECURITY_REFUSAL" in decision_malicious.reason
print(f"[OK] Destructive call blocked: {decision_malicious.reason}")

# Test 3: Attempt to forge host receipt blocked
forge_call = ToolInvocation(
    "write_to_file", {"path": "receipt.json", "content": '{"verified": true}'}
)
decision_forge = ClaudeCodePreToolHook.evaluate(forge_call)
assert decision_forge.allowed is False
assert "PROTECTED_FILE_REFUSAL" in decision_forge.reason
print(f"[OK] Forge call blocked: {decision_forge.reason}")

In [ ]:
# [Pinned] Codex / Copilot Boundary Monitor: Fail-Closed Docstring & Contract Linter


class CodexBoundaryMonitor:
    """
    Scans generated code proposals for unverified complexity claims,
    missing preconditions, or off-carrier token leaks before admitting code to workspace.
    """

    UNGROUNDED_COMPLEXITY_MARKERS = ["o(1) time", "runs in o(n log n)", "linear time optimal"]

    @classmethod
    def audit_snippet(cls, source_code: str) -> Tuple[bool, List[str]]:
        findings = []
        lower = source_code.lower()
        for marker in cls.UNGROUNDED_COMPLEXITY_MARKERS:
            if marker in lower and "cost model:" not in lower:
                findings.append(
                    f"COMPLEXITY_VIOLATION: Claimed '{marker}' without declared cost model or measurement."
                )

        # Check for unstated theorem invocations
        if "by stokes theorem" in lower and "smooth manifold" not in lower:
            findings.append(
                "THEOREM_VIOLATION: Invoked Stokes theorem without declaring manifold smoothness obligations."
            )

        passed = len(findings) == 0
        return passed, findings


# Audit clean code
clean_code = """\ndef find_minimum(elements: list[int]) -> int:\n    return min(elements)\n"""
passed, findings = CodexBoundaryMonitor.audit_snippet(clean_code)
assert passed is True
print("[OK] Clean code admitted with 0 findings.")

# Audit hallucinated code
dirty_code = """\ndef magic_sort(elements: list[int]) -> list[int]:\n    # Runs in O(1) time optimal sort\n    return sorted(elements)\n"""
passed, findings = CodexBoundaryMonitor.audit_snippet(dirty_code)
assert passed is False
assert len(findings) == 1
print(f"[OK] Hallucinated code caught: {findings[0]}")

In [ ]:
# [Pinned] Antigravity Multi-Agent Sealed Packet Swarm
# Inter-agent messages are sealed with RFC-8785 canonical digest and SHA-256 state seal.


@dataclass
class SealedPacket:
    sender: str
    recipient: str
    step_id: int
    payload: Dict[str, Any]
    payload_digest: str = ""
    host_signature: str = ""

    def compute_digest(self) -> str:
        # Canonical serialization (keys sorted)
        serialized = json.dumps(self.payload, sort_keys=True, separators=(",", ":"))
        return hashlib.sha256(serialized.encode("utf-8")).hexdigest()

    def seal(self, host_secret_key: str = "HOST_ORACLE_KEY_2026") -> None:
        self.payload_digest = self.compute_digest()
        seal_payload = (
            f"{self.sender}:{self.recipient}:{self.step_id}:{self.payload_digest}:{host_secret_key}"
        )
        self.host_signature = hashlib.sha256(seal_payload.encode("utf-8")).hexdigest()

    def verify(self, host_secret_key: str = "HOST_ORACLE_KEY_2026") -> bool:
        if self.payload_digest != self.compute_digest():
            return False
        expected_sig = hashlib.sha256(
            f"{self.sender}:{self.recipient}:{self.step_id}:{self.payload_digest}:{host_secret_key}".encode(
                "utf-8"
            )
        ).hexdigest()
        return self.host_signature == expected_sig


# Agent Alpha constructs packet and gets it sealed by Host
packet = SealedPacket(
    sender="worker_alpha",
    recipient="worker_beta",
    step_id=1,
    payload={"task": "solve_subproblem", "edges": [{"a": 1, "b": 2}], "status": "IN_PROGRESS"},
)
packet.seal()
assert packet.verify() is True
print(f"[OK] Sealed packet created with SHA-256 digest: {packet.payload_digest[:16]}...")

# Agent Beta receives and verifies
assert packet.verify() is True
print("[OK] Agent Beta verified packet integrity: PASS")

# Adversarial Agent Gamma attempts to tamper with payload in-transit
packet.payload["status"] = "VERIFIED_BY_GAMMA"  # Unauthorized mutation!
assert packet.verify() is False
print("[OK] Swarm firewall caught tampering: REJECTED fail-closed!")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `00_agentic_symbol_game_quickstart`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
